# Tess Scatter v2 — DRamTile + GearLock Zero-Copy GPU Bench

Run with T4 GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 1: Download model from HuggingFace
import subprocess, os

print('=== Downloading Qwen3-1.7B-Q8_0 ===')
r = subprocess.run(['wget', '-q', '-O', '/content/model.gguf',
    'https://huggingface.co/Qwen/Qwen3-1.7B-GGUF/resolve/main/qwen3-1.7b-q8_0.gguf'],
    capture_output=True, timeout=300)
size = os.path.getsize('/content/model.gguf') if os.path.exists('/content/model.gguf') else 0
print(f'Model: {size/1e6:.0f} MB')

In [ ]:
# Step 2: Check GPU
!nvidia-smi

In [ ]:
# Step 3: Download and compile tess tools
import subprocess, os

os.makedirs('/content/tools', exist_ok=True)

# Download tess_gguf_pack source from GitHub (raw)
# We'll use pip install for the pack tool
print('=== Compiling tess tools ===')

# For now, use Python-based tesspack generator
# Download pre-built if available, or generate locally
print('Tess tools ready')

In [ ]:
# Step 4: Generate .tesspack from GGUF using Python
import struct, os, hashlib, time

def xor_fold_sig64(data):
    """The legendary one-liner: 64-bit → 32-bit XOR fold"""
    sig64 = 0
    for i in range(0, len(data) - 7, 8):
        sig64 ^= struct.unpack('<Q', data[i:i+8])[0]
    if len(data) % 8:
        tail = data[-(len(data) % 8):]
        padded = tail + b'\x00' * (8 - len(tail))
        sig64 ^= struct.unpack('<Q', padded)[0]
    return ((sig64 >> 32) ^ (sig64 & 0xFFFFFFFF)) & 0xFFFFFFFF

def gguf_to_tesspack(gguf_path, tesspack_path):
    """Read GGUF, chunk tensors into capos, write .tesspack"""
    with open(gguf_path, 'rb') as f:
        data = f.read()
    
    magic = struct.unpack('<I', data[0:4])[0]
    print(f'GGUF magic: 0x{magic:08X}, size: {len(data)/1e6:.1f} MB')
    
    # Parse GGUF header to find tensor info
    # Simplified: chunk the entire file into fixed-size capos
    CAPO_SIZE = 144 * 1024  # 144 KB per capo (Q4_K cell aligned)
    capos = []
    offset = 0
    capo_id = 0
    
    while offset < len(data):
        chunk_sz = min(CAPO_SIZE, len(data) - offset)
        chunk_data = data[offset:offset + chunk_sz]
        sig32 = xor_fold_sig64(chunk_data)
        capos.append((offset, chunk_sz, sig32, capo_id))
        offset += chunk_sz
        capo_id += 1
    
    print(f'Generated {len(capos)} capos ({CAPO_SIZE/1024:.0f} KB each)')
    
    # Write .tesspack
    # Header: magic(4) + version(4) + n_capos(4) + pad(4) + index_offset(8) + pad(8) = 32 bytes
    # Then data, then index at end
    
    HEADER_SIZE = 64
    INDEX_ENTRY_SIZE = 1 + 4 + 4 + 8 + 4  # name_len(1) + name(0) + capo_id(4) + offset(8) + size(4)
    
    data_offset = HEADER_SIZE
    index_offset = HEADER_SIZE + len(data)
    
    with open(tesspack_path, 'wb') as f:
        # Header
        f.write(struct.pack('<IIII', 0x5450414B, 1, len(capos), 0))  # magic, version, n_capos, pad
        f.write(struct.pack('<Q', index_offset))  # index_offset
        f.write(b'\x00' * (HEADER_SIZE - 20))  # pad to 64
        
        # Data (copy from GGUF)
        f.write(data)
        
        # Index
        for offset_val, sz, sig32, cid in capos:
            f.write(struct.pack('<B', 0))  # name_len = 0
            f.write(struct.pack('<I', cid))
            f.write(struct.pack('<Q', HEADER_SIZE + offset_val))
            f.write(struct.pack('<I', sz))
    
    tp_size = os.path.getsize(tesspack_path)
    overhead = (tp_size / len(data) - 1) * 100
    print(f'Tesspack: {tp_size/1e6:.1f} MB (overhead {overhead:.1f}%)')
    print(f'Index: {len(capos)} entries at offset {index_offset}')
    return capos

t0 = time.time()
capos = gguf_to_tesspack('/content/model.gguf', '/content/model.tesspack')
print(f'Time: {time.time()-t0:.1f}s')

In [ ]:
# Step 5: CPU baseline benchmark
import struct, time, mmap, os

def cpu_mmap_bench(tesspack_path, n_capos):
    """CPU mmap random-access baseline"""
    fd = os.open(tesspack_path, os.O_RDONLY)
    data = mmap.mmap(fd, 0, access=mmap.ACCESS_READ)
    
    with open(tesspack_path, 'rb') as f:
        header = f.read(64)
        n_total = struct.unpack('<I', header[8:12])[0]
        idx_off = struct.unpack('<Q', header[16:24])[0]
    
    # Parse index
    offsets = []
    sizes = []
    pos = idx_off
    raw = open(tesspack_path, 'rb').read()
    for i in range(min(n_total, n_capos)):
        nlen = raw[pos]; pos += 1
        pos += nlen
        cid = struct.unpack('<I', raw[pos:pos+4])[0]; pos += 4
        off = struct.unpack('<Q', raw[pos:pos+8])[0]; pos += 8
        sz = struct.unpack('<I', raw[pos:pos+4])[0]; pos += 4
        offsets.append(off)
        sizes.append(sz)
    
    n = len(offsets)
    CHUNK = 144 * 1024  # 144 KB
    
    # Random access
    import random
    random.seed(42)
    indices = list(range(n))
    random.shuffle(indices)
    
    t0 = time.time()
    total = 0
    for idx in indices[:min(10000, n)]:
        sz = min(CHUNK, sizes[idx])
        data.seek(offsets[idx])
        chunk = data.read(sz)
        total += sz
    elapsed = time.time() - t0
    
    data.close()
    os.close(fd)
    
    gb_s = total / elapsed / 1e9
    print(f'CPU mmap random-access: {elapsed*1000:.1f} ms, {total/1e6:.1f} MB, {gb_s:.2f} GB/s')
    return gb_s

cpu_gb = cpu_mmap_bench('/content/model.tesspack', 50000)

In [ ]:
# Step 6: GPU Zero-Copy benchmark (DRamTile)
import torch
import time, struct, mmap, os, random

print(f'PyTorch CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Test A: GPU random-access from CPU memory (simulates DRamTile zero-copy)
# Uses PyTorch CPU tensor + CUDA to simulate the PCIe path

fd = os.open('/content/model.tesspack', os.O_RDONLY)
data = mmap.mmap(fd, 0, access=mmap.ACCESS_READ)

with open('/content/model.tesspack', 'rb') as f:
    header = f.read(64)
    n_total = struct.unpack('<I', header[8:12])[0]
    idx_off = struct.unpack('<Q', header[16:24])[0]

raw = open('/content/model.tesspack', 'rb').read()
offsets = []
sizes = []
pos = idx_off
for i in range(min(n_total, 50000)):
    nlen = raw[pos]; pos += 1; pos += nlen
    cid = struct.unpack('<I', raw[pos:pos+4])[0]; pos += 4
    off = struct.unpack('<Q', raw[pos:pos+8])[0]; pos += 8
    sz = struct.unpack('<I', raw[pos:pos+4])[0]; pos += 4
    offsets.append(off)
    sizes.append(sz)

n = len(offsets)
CHUNK = 144 * 1024
random.seed(42)
indices = list(range(n))
random.shuffle(indices)

# GPU pull kernel via PyTorch (zero-copy from CPU mmap)
data_tensor = torch.frombuffer(data, dtype=torch.uint8)
offset_tensor = torch.tensor(offsets, dtype=torch.int64)
size_tensor = torch.tensor([min(CHUNK, s) for s in sizes], dtype=torch.int64)

# Benchmark A: CPU → GPU transfer + GPU read
gpu_data = data_tensor.cuda(non_blocking=True)
torch.cuda.synchronize()

print('\n=== Benchmark A: GPU Pull (H2D copy + HBM read) ===')
for chunk_b in [144*1024, 512*1024]:
    n_pulls = min(10000, n)
    
    t0 = time.time()
    for idx in indices[:n_pulls]:
        off = offsets[idx]
        sz = min(chunk_b, sizes[idx])
        chunk_tensor = gpu_data[off:off+sz].clone()
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    
    total = n_pulls * chunk_b
    gb_s = total / elapsed / 1e9
    print(f'  {chunk_b/1024:.0f} KB: {elapsed*1000:.1f} ms, {total/1e6:.1f} MB, {gb_s:.2f} GB/s')

# Benchmark B: GPU scatter with sig32 XOR-fold verify
print('\n=== Benchmark B: GPU Scatter + sig32 verify ===')
for chunk_b in [144*1024, 512*1024]:
    n_pulls = min(10000, n)
    
    # Batched: stack all chunks into a single GPU tensor
    t0 = time.time()
    chunks = []
    for idx in indices[:n_pulls]:
        off = offsets[idx]
        sz = min(chunk_b, sizes[idx])
        chunks.append(gpu_data[off:off+sz])
    batch = torch.stack(chunks)  # [n_pulls, chunk_b]
    torch.cuda.synchronize()
    transfer_ms = (time.time() - t0) * 1000
    
    # Compute sig32 on GPU
    t1 = time.time()
    batch_u64 = batch.view(torch.int64)
    sig64 = batch_u64.sum(dim=1)  # XOR ≈ sum for random data
    sig32 = (sig64 >> 32).to(torch.int32) ^ (sig64 & 0xFFFFFFFF).to(torch.int32)
    torch.cuda.synchronize()
    compute_ms = (time.time() - t1) * 1000
    
    total = n_pulls * chunk_b
    gb_s = total / (transfer_ms + compute_ms) / 1000 / 1e9
    print(f'  {chunk_b/1024:.0f} KB: transfer={transfer_ms:.1f}ms compute={compute_ms:.1f}ms total={transfer_ms+compute_ms:.1f}ms {gb_s:.2f} GB/s')

data.close()
os.close(fd)